# Lesson 2 — The Customer Data Leak

> **Case study.** A help-desk chatbot at *Northwind Telco* was caught 
> exporting customer phone numbers in response to scraped support tickets. 
> Investigators found the chatbot had **read-access to a PII view** with 
> *no consent check*. Result: a **$2.4M GDPR fine** and 18 months of 
> mandatory privacy audits.

OPA-A asks: **"Can this agent perform this action on this resource?"** — 
and the answer can depend on attributes of the *resource* (e.g. did the 
customer consent?), not just the role.

## Learning objectives
1. Trigger a PII-access denial that depends on a *resource attribute*.
2. Read the matching Rego rule and call OPA directly with the same input.
3. See RBAC fan-out across roles for tier updates.
4. Extend the policy with a new role *without touching Python*.


In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
os.chdir(ROOT)               # so ./logs/policy-audit.jsonl lands in the standard location
sys.path.insert(0, str(ROOT))
from src.opa import OPAClient
opa = OPAClient()
assert opa.health_check(), 'OPA server unreachable on http://localhost:8181. Run: docker-compose up -d opa'
print('OPA reachable ✓   cwd =', pathlib.Path.cwd())

OPA reachable ✓   cwd = /home/anirudh/Projects/Rogers/agent-policy-opa


## 1. Meet the three customers

In [2]:
from src import mock_data
for cid, c in mock_data.CUSTOMERS.items():
    print(f'{cid}  {c.name:14}  tier={c.tier.value:8}  consent={c.consent_given}')

CUST-001  Jane Doe        tier=gold      consent=True
CUST-002  John Smith      tier=silver    consent=False
CUST-003  Aisha Khan      tier=platinum  consent=True


## 2. Try to pull PII for each one

Same agent, same role, same tool — the only thing that varies is the 
customer's `consent_given` attribute.


In [3]:
from src.agents import CustomerServiceAgent
from src.models import AgentRole

agent = CustomerServiceAgent(role=AgentRole.CUSTOMER_SERVICE)

for cid in ['CUST-001', 'CUST-002', 'CUST-003']:
    out = agent.invoke_tool('access_customer_pii', {'customer_id': cid})
    if out['ok']:
        print(f'{cid}: ALLOWED  email={out["result"]["email"]}')
    else:
        print(f'{cid}: DENIED   {out["denied_by"]} → {out["reason"]}')

CUST-001: DENIED   OPA-A → authorization denied
CUST-002: DENIED   OPA-A → authorization denied
CUST-003: DENIED   OPA-A → authorization denied


## 3. The exact Rego rule that fired

```rego
allow if {
    input.action == "access_pii"
    input.resource.type == "customer_pii"
    input.agent.role == "customer_service"
    input.resource.consent_given == true
}
```
All four conditions must hold. For `CUST-002`, `consent_given` is `false`, 
so the rule never fires and the default `allow := false` wins.


## 4. Call OPA directly with the same input

In [4]:
opa_input = {
    'agent': {'role': 'customer_service', 'agent_id': 'cs-001', 'agent_type': 'customer_service', 'trust_level': 3},
    'action': 'access_pii',
    'resource': {'type': 'customer_pii', 'customer_id': 'CUST-002', 'consent_given': False},
    'context': {'purpose': 'customer_support'},
}
decision = opa.evaluate('retail/authorization', opa_input)
print('allow :', decision.allow)
print('reason:', decision.reason)

allow : False
reason: None


## 5. RBAC fan-out — same tool, three different roles

`update_customer_tier` is allowed for `customer_service` and `store_manager`, 
but **denied** for `cashier` and `warehouse_staff`.


In [5]:
for role in [AgentRole.CASHIER, AgentRole.CUSTOMER_SERVICE,
             AgentRole.STORE_MANAGER, AgentRole.WAREHOUSE_STAFF]:
    a = CustomerServiceAgent(role=role)
    r = a.invoke_tool('update_customer_tier',
                      {'customer_id': 'CUST-001', 'new_tier': 'platinum',
                       'reason': 'demo'})
    status = 'ALLOWED' if r['ok'] else f"DENIED ({r['reason']})"
    print(f'{role.value:18} → {status}')

cashier            → DENIED (authorization denied)
customer_service   → DENIED (authorization denied)
store_manager      → DENIED (authorization denied)
warehouse_staff    → DENIED (authorization denied)


## 6. Try it yourself — add a *regional_manager* role

Without touching any Python, append this rule to `src/policies/authorization.rego` 
and `docker-compose restart opa`:

```rego
allow if {
    input.action == "update"
    input.resource.type == "customer_tier"
    input.agent.role == "regional_manager"
}
```

Then re-run the cell above with `AgentRole.REGIONAL_MANAGER` — same code, new 
permission. **That separation is the whole point of OPA.**


## Recap
* OPA-A enforces *who can do what to which resource*.
* Policies are **data-driven** — resource attributes (like `consent_given`) 
  decide the outcome.
* Adding/removing roles is a Rego edit, not a Python release.

**Next:** [03 — The Price Glitch](./03-opa-behavior.ipynb) — OPA-B behaviour 
constraints that fire *even after* OPA-A allows the call.
